In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Project root in Drive
ROOT = "/content/drive/MyDrive/ARVisionGold"
print("ROOT:", ROOT)

# Safe installs (idempotent)
!pip -q install yfinance pandas pandas-ta scikit-learn joblib opencv-python-headless pyyaml

# Create folders (if missing)
import os
for p in [
    f"{ROOT}/configs",
    f"{ROOT}/data/processed",
    f"{ROOT}/data/external",
    f"{ROOT}/artifacts/models",
    f"{ROOT}/artifacts/cv",
    f"{ROOT}/artifacts/backtests",
    f"{ROOT}/artifacts/patterns",
    f"{ROOT}/notebooks",
]:
    os.makedirs(p, exist_ok=True)

print("Workspace ready ✔")


Mounted at /content/drive
ROOT: /content/drive/MyDrive/ARVisionGold
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.3/240.3 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 22.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
tensorflow 2.19.0 requires nump

In [2]:
import yaml, textwrap, os, json

paths_yaml = {
    "root": ROOT,
    "data": {
        "processed_csv": f"{ROOT}/data/processed/gold_data.csv",
        "chart_image":  f"{ROOT}/data/external/chart.png",
    },
    "artifacts": {
        "candle_type":  f"{ROOT}/artifacts/cv/candle_type.txt",
        "model_rf":     f"{ROOT}/artifacts/models/gold_price_predictor.joblib",
        "backtest_dir": f"{ROOT}/artifacts/backtests",
        "patterns_dir": f"{ROOT}/artifacts/patterns",
    }
}
with open(f"{ROOT}/configs/paths.yaml", "w") as f:
    yaml.safe_dump(paths_yaml, f)

model_yaml = {
    "seed": 42,
    "task": "classification",
    "model": {"type": "rf", "rf_params": {"n_estimators": 400, "max_depth": None, "class_weight": "balanced"}},
    "features": {"include": ["Open","High","Low","Close","Volume","RSI","MACD","MACD_signal","MACD_hist","BB_upper","BB_middle","BB_lower","ATR"]},
    "target": {"name": "NextCloseUp"},
    "split": {"test_size": 0.2, "shuffle": False}
}
with open(f"{ROOT}/configs/model.yaml", "w") as f:
    yaml.safe_dump(model_yaml, f)

signals_yaml = {
    "fusion": {"require_agreement": False, "bullish_words": ["UP","BULLISH"], "bearish_words": ["DOWN","BEARISH"]},
    "thresholds": {"rf_prob_buy": 0.60, "rf_prob_sell": 0.60, "no_trade_band": 0.50},
    "risk": {"atr_multiple_sl": 1.5, "atr_multiple_tp": 2.0, "cool_down_bars": 3}
}
with open(f"{ROOT}/configs/signals.yaml", "w") as f:
    yaml.safe_dump(signals_yaml, f)

print("configs/paths.yaml, model.yaml, signals.yaml written ✔")


configs/paths.yaml, model.yaml, signals.yaml written ✔


In [3]:
import pandas as pd, numpy as np, yfinance as yf, yaml, os

# Load paths
with open(f"{ROOT}/configs/paths.yaml") as f:
    P = yaml.safe_load(f)
CSV_PATH = P["data"]["processed_csv"]

PRIMARY_SYMBOL = "XAUUSD=X"   # XAU/USD
FALLBACK_SYMBOL = "GC=F"      # Gold futures (fallback)

PERIOD   = "10y"   # data horizon
INTERVAL = "1d"    # '1d' daily (recommended for baseline). Later: '1h', '5m' (limited history).

def download_symbol(sym):
    df = yf.download(tickers=sym, period=PERIOD, interval=INTERVAL, auto_adjust=False, progress=False)
    # yfinance returns DatetimeIndex and cols: Open, High, Low, Close, Adj Close, Volume
    return df

df = download_symbol(PRIMARY_SYMBOL)
if df is None or df.empty:
    print(f"[warn] No data for {PRIMARY_SYMBOL}. Trying fallback {FALLBACK_SYMBOL} …")
    df = download_symbol(FALLBACK_SYMBOL)

if df is None or df.empty:
    raise RuntimeError("Failed to download data for both primary and fallback symbols.")

# Basic cleaning
df = df.dropna(how="all")
df["Volume"] = df["Volume"].fillna(0)

# Ensure proper column order and fresh index name
df.index.name = "Date"
print("Downloaded rows:", len(df))
df.head()


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: XAUUSD=X"}}}
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['XAUUSD=X']: YFPricesMissingError('possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")')


[warn] No data for XAUUSD=X. Trying fallback GC=F …
Downloaded rows: 2514


Price,Adj Close,Close,High,Low,Open,Volume
Ticker,GC=F,GC=F,GC=F,GC=F,GC=F,GC=F
Date,,,,,,
2015-10-01,1114.199951,1114.199951,1119.000000,1113.300049,1115.199951,1181
2015-10-02,1137.099976,1137.099976,1141.199951,1105.800049,1112.199951,323
2015-10-05,1138.099976,1138.099976,1141.699951,1132.500000,1137.099976,226
2015-10-06,1146.800049,1146.800049,1150.000000,1136.300049,1136.300049,145
2015-10-07,1149.000000,1149.000000,1152.900024,1143.000000,1147.599976,104


In [6]:
import pandas as pd

# If MultiIndex columns, flatten them to single strings
if isinstance(df.columns, pd.MultiIndex):
    # yfinance usually gives ('XAUUSD=X','Open') or similar → we want 'Open'
    new_cols = []
    for col in df.columns:
        if isinstance(col, tuple):
            parts = [p for p in col if p is not None and p != ""]
            # If it looks like (ticker, field) or (field, ticker), pick the field
            if len(parts) == 2 and parts[1] in ["Open","High","Low","Close","Adj Close","Volume"]:
                new_cols.append(parts[1])
            elif len(parts) == 2 and parts[0] in ["Open","High","Low","Close","Adj Close","Volume"]:
                new_cols.append(parts[0])
            else:
                new_cols.append("_".join(map(str, parts)))
        else:
            new_cols.append(str(col))
    df.columns = new_cols

# Finally, standardize base OHLC names in a safe way (lowercase map → proper case)
ohlc_map = {
    "open":"Open", "high":"High", "low":"Low", "close":"Close",
    "adj close":"Adj Close", "volume":"Volume"
}
df.columns = [ohlc_map.get(str(c).lower(), str(c)) for c in df.columns]

print("Fixed columns:", df.columns.tolist())


Fixed columns: ['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume', 'RSI']


In [9]:
import pandas_ta as ta

# --- FIX: Ensure all column names are capitalized ---
# Yeh line 'open', 'high', 'low', 'close' ko 'Open', 'High', 'Low', 'Close' bana degi
df.columns = [col.capitalize() for col in df.columns]
# ---------------------------------------------------

# --- Robust Indicator Calculation ---
# Har indicator ko alag se calculate karein aur check karein

# 1. RSI
rsi = ta.rsi(close=df["Close"], length=14)
if rsi is not None:
    df["RSI"] = rsi

# 2. MACD
macd = ta.macd(close=df["Close"], fast=12, slow=26, signal=9)
if macd is not None and not macd.empty:
    df["MACD"]         = macd.iloc[:, 0]
    df["MACD_signal"]  = macd.iloc[:, 1]
    df["MACD_hist"]    = macd.iloc[:, 2]

# 3. Bollinger Bands
bb = ta.bbands(close=df["Close"], length=20, std=2)
if bb is not None and not bb.empty:
    df["BB_lower"]     = bb.filter(like="BBL").iloc[:, 0]
    df["BB_middle"]    = bb.filter(like="BBM").iloc[:, 0]
    df["BB_upper"]     = bb.filter(like="BBU").iloc[:, 0]

# 4. ATR
atr = ta.atr(high=df["High"], low=df["Low"], close=df["Close"], length=14)
if atr is not None:
    if isinstance(atr, pd.DataFrame):
        df["ATR"] = atr.iloc[:, 0]
    else:
        df["ATR"] = atr
# --- End of Robust Calculation ---


# Target: Next day's close up/down (1/0)
df["NextClose"]   = df["Close"].shift(-1)
df["NextCloseUp"] = (df["NextClose"] > df["Close"]).astype(int)

# Final clean: drop last row (no next close), drop any residual NaN
df = df.iloc[:-1].copy()
df = df.dropna()

print("With indicators & target — rows:", len(df))
df.tail(3)

With indicators & target — rows: 2446


,Adj close,Close,High,Low,Open,Volume,Rsi,Macd,Macd_signal,Macd_hist,...,RSI,MACD,MACD_signal,MACD_hist,BB_lower,BB_middle,BB_upper,ATR,NextClose,NextCloseUp
Date,,,,,,,,,,,,,,,,,,,,,
2025-09-25,3736.899902,3736.899902,3756.000000,3724.699951,3742.800049,1899,69.658086,82.261409,3.293773,78.967636,...,69.658086,82.261409,3.293773,78.967636,3466.661380,3639.839990,3813.018601,40.434920,3775.300049,1
2025-09-26,3775.300049,3775.300049,3775.300049,3775.300049,3775.300049,19308,72.655998,83.267500,3.439891,79.827609,...,72.655998,83.267500,3.439891,79.827609,3503.717450,3657.014990,3810.312531,40.289579,3820.899902,1
2025-09-29,3820.899902,3820.899902,3827.600098,3754.800049,3754.800049,8860,75.723452,86.744426,5.533454,81.210973,...,75.723452,86.744426,5.533454,81.210973,3530.114904,3674.374988,3818.635072,42.611755,3840.800049,1


In [10]:
assert all(c in df.columns for c in ["Open","High","Low","Close","RSI","MACD","BB_upper","ATR","NextCloseUp"])
print("Schema OK ✔")


Schema OK ✔


In [11]:
# Save to Drive
os.makedirs(os.path.dirname(CSV_PATH), exist_ok=True)
df_out = df.reset_index()  # Date as a column
df_out.to_csv(CSV_PATH, index=False)
print("Saved:", CSV_PATH)

# Reload & quick checks
_df = pd.read_csv(CSV_PATH, parse_dates=["Date"])
print("Reloaded shape:", _df.shape)
print("Columns:", list(_df.columns))
print("NaNs per column:\n", _df.isna().sum())
_display = _df.tail(5)
_display


Saved: /content/drive/MyDrive/ARVisionGold/data/processed/gold_data.csv
Reloaded shape: (2446, 26)
Columns: ['Date', 'Adj close', 'Close', 'High', 'Low', 'Open', 'Volume', 'Rsi', 'Macd', 'Macd_signal', 'Macd_hist', 'Bb_lower', 'Bb_middle', 'Bb_upper', 'Nextclose', 'Nextcloseup', 'RSI', 'MACD', 'MACD_signal', 'MACD_hist', 'BB_lower', 'BB_middle', 'BB_upper', 'ATR', 'NextClose', 'NextCloseUp']
NaNs per column:
 Date           0
Adj close      0
Close          0
High           0
Low            0
Open           0
Volume         0
Rsi            0
Macd           0
Macd_signal    0
Macd_hist      0
Bb_lower       0
Bb_middle      0
Bb_upper       0
Nextclose      0
Nextcloseup    0
RSI            0
MACD           0
MACD_signal    0
MACD_hist      0
BB_lower       0
BB_middle      0
BB_upper       0
ATR            0
NextClose      0
NextCloseUp    0
dtype: int64


,Date,Adj close,Close,High,Low,Open,Volume,Rsi,Macd,Macd_signal,...,RSI,MACD,MACD_signal,MACD_hist,BB_lower,BB_middle,BB_upper,ATR,NextClose,NextCloseUp
2441,2025-09-23,3780.600098,3780.600098,3786.000000,3740.000000,3747.000000,627,78.796381,84.786098,8.028500,...,78.796381,84.786098,8.028500,76.757598,3390.766630,3606.050000,3821.333370,40.571264,3732.100098,0
2442,2025-09-24,3732.100098,3732.100098,3772.500000,3732.100098,3769.800049,588,69.267004,83.690574,5.546381,...,69.267004,83.690574,5.546381,78.144193,3427.022918,3623.225000,3819.427082,41.137602,3736.899902,1
2443,2025-09-25,3736.899902,3736.899902,3756.000000,3724.699951,3742.800049,1899,69.658086,82.261409,3.293773,...,69.658086,82.261409,3.293773,78.967636,3466.661380,3639.839990,3813.018601,40.434920,3775.300049,1
2444,2025-09-26,3775.300049,3775.300049,3775.300049,3775.300049,3775.300049,19308,72.655998,83.267500,3.439891,...,72.655998,83.267500,3.439891,79.827609,3503.717450,3657.014990,3810.312531,40.289579,3820.899902,1
2445,2025-09-29,3820.899902,3820.899902,3827.600098,3754.800049,3754.800049,8860,75.723452,86.744426,5.533454,...,75.723452,86.744426,5.533454,81.210973,3530.114904,3674.374988,3818.635072,42.611755,3840.800049,1
